In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

from datetime import datetime, timedelta
from random import randint, random, seed

from engines import GAMealPlanner, get_pantry_ingredient, load_all_ingredients, make_preferences
from models import (
    MealPlanningEnvironment,
    Pantry,
)

In [2]:
seed(1)

In [3]:
preferences = make_preferences()

In [4]:
all_ingredients = load_all_ingredients("../recipe_extraction/supplemented_structured_ingredients.json")

In [5]:
CURRENT_DATE = datetime.now()

In [6]:
pantry_ingredient_name_by_quantity = {
    "chicken breast": 800,
    "broccoli": 1500,
    "rice": 1000,
}

In [7]:
pantry_ingredients = [
    get_pantry_ingredient(name, CURRENT_DATE + timedelta(days=randint(1, 7)), all_ingredients)
    for name in pantry_ingredient_name_by_quantity.keys()
]

In [8]:
pantry_ingredient_by_quantity = dict(zip(pantry_ingredients, pantry_ingredient_name_by_quantity.values()))

In [9]:
pantry = Pantry()

for pantry_ingredient, quantity in pantry_ingredient_by_quantity.items():
    pantry.add(
        pantry_ingredient,
        quantity,
    )

In [10]:
pantry.print()

---
Quantity: 800 g
Ingredient: chicken breast
	Estimated Expiration Date: 2026-04-30
	Nutritional Information:
		Calories: 125.0 kcal
		Carbohydrates: 1.79 g
		Sugar: 0.0 g
		Protein: 16.07 g
		Fat: 5.36 g
		Saturated Fat: 1.79 g
		Fiber: 1.8 g
		Sodium: 571.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: No
		Vegan: No
---
---
Quantity: 1500 g
Ingredient: broccoli
	Estimated Expiration Date: 2026-05-03
	Nutritional Information:
		Calories: 31.0 kcal
		Carbohydrates: 6.27 g
		Sugar: 1.4 g
		Protein: 2.57 g
		Fat: 0.34 g
		Saturated Fat: 0.039 g
		Fiber: 2.4 g
		Sodium: 36.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: Yes
		Vegan: Yes
---
---
Quantity: 1000 g
Ingredient: rice
	Estimated Expiration Date: 2026-05-05
	Nutritional Information:
		Calories: 356.0 kcal
		Carbohydrates: 80.0 g
		Sugar: 0.0 g
		Protein: 6.67 g
		Fat: 0.0 g
		Saturated Fat: 0.0 g
		Fiber: 2.2 g
		Sodium: 0.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: Yes
		Vegan: Yes
---


In [11]:
meal_planning_environment = MealPlanningEnvironment(
    recipes=[],
    pantry=pantry,
    preferences=preferences,
)

In [12]:
meal_planning_environment.load_recipes_from_json("../recipe_extraction/supplemented_structured_recipes.json")

In [13]:
all_ingredient_names = []

for recipe in meal_planning_environment.recipes:
    for ingredient_name in recipe.ingredients.keys():
        all_ingredient_names.append(ingredient_name)

In [14]:
all_ingredient_costs = {}

for ingredient_name in sorted(set(all_ingredient_names)):
    all_ingredient_costs[ingredient_name] = random()

In [15]:
meal_planning_environment.ingredient_costs = all_ingredient_costs
meal_planning_environment._check_ingredient_costs()

In [16]:
planner = GAMealPlanner(meal_planning_environment)

In [17]:
NUM_GENERATIONS = 500
NUM_PARENTS_MATING = 20
POPULATION_SIZE = 100
NUM_DAYS = 7
NUM_MEALS_PER_DAY = 3
GENERATION_PRINT_INTERVAL = 10
SEED = 1

In [18]:
best_meal_plan, best_fitness_score = planner.generate_meal_plan(
    num_days=NUM_DAYS,
    meals_per_day=NUM_MEALS_PER_DAY,
    num_generations=NUM_GENERATIONS,
    num_parents_mating=NUM_PARENTS_MATING,
    population_size=POPULATION_SIZE,
    generation_print_interval=GENERATION_PRINT_INTERVAL,
    seed=SEED,
)

Generation 10, Best Fitness: -33.48
Generation 20, Best Fitness: -29.51
Generation 30, Best Fitness: -22.45
Generation 40, Best Fitness: -21.57
Generation 50, Best Fitness: -20.44
Generation 60, Best Fitness: -18.67
Generation 70, Best Fitness: -17.72
Generation 80, Best Fitness: -17.20
Generation 90, Best Fitness: -17.20
Generation 100, Best Fitness: -16.60
Generation 110, Best Fitness: -16.28
Generation 120, Best Fitness: -16.28
Generation 130, Best Fitness: -16.04
Generation 140, Best Fitness: -16.04
Generation 150, Best Fitness: -16.04
Generation 160, Best Fitness: -16.04
Generation 170, Best Fitness: -16.04
Generation 180, Best Fitness: -15.78
Generation 190, Best Fitness: -15.78
Generation 200, Best Fitness: -15.78
Generation 210, Best Fitness: -15.59
Generation 220, Best Fitness: -15.59
Generation 230, Best Fitness: -15.59
Generation 240, Best Fitness: -15.59
Generation 250, Best Fitness: -15.59
Generation 260, Best Fitness: -15.57
Generation 270, Best Fitness: -15.57
Generation

In [24]:
print(f"Best fitness score: {best_fitness_score:.2f}")

Best fitness score: -7.49


In [25]:
pantry_consumption_df = planner.get_pantry_consumption()
pantry_consumption_df

,Ingredient,Available,Consumed,Unused,Expires in
0,chicken breast,800,0.000000,800.000000,1d
1,broccoli,1500,566.990463,933.009537,4d
2,rice,1000,0.000000,1000.000000,6d


In [26]:
meal_plan_df = planner.get_meal_plan_recipes()
meal_plan_df

,Day,Meal 1,Meal 2,Meal 3
0,1,Frozen Meyer Lemon Cream with Blackberry Sauce,Pear-Caramel Ice Cream Sundaes,Jalapeño-Monterey Jack Grits
1,2,Smoked Gouda Grits,"Slow-Roasted Salmon With Fennel, Citrus, and C...",Butter Pie Crust
2,3,Cherry Blossom,Roasted Broccoli Florets with Toasted Breadcru...,Sweet Potato Macaroni and Cheese
3,4,"Yellow and Green Bean Salad with Olives, Cherr...",Cream of Broccoli Soup,Matzo Scallion Pancakes
4,5,Lemon Mascarpone Brûlée Tarts with Raspberry S...,Orange Duck Breasts on Braised Chicory,Modern Mince Pie
5,6,Pineapple Anise Sherbet,Simple Vegetable Couscous,Chocolate-Cranberry Torte
6,7,Pumpkin Cake with Sage Ice Cream and Pumpkin C...,Fresh Corn Pancakes,Herb-Crusted Grilled Lamb with Apricot Relish


In [27]:
shopping_list_df, num_ingredients, total_cost = planner.get_shopping_list()

print(f"Total ingredients needed to purchase: {num_ingredients}")
print(f"Total cost: €{total_cost:.2f}")

shopping_list_df

Total ingredients needed to purchase: 147
Total cost: €39.67


,Ingredient,Quantity to Buy (g),Cost (€)
0,Accompaniment,27.1,0.17
1,Buttermilk Pie Crust Dough disks,33.3,0.16
2,Chambord,9.9,0.06
3,English mustard,1.0,0.00
4,Flaky sea salt,33.3,0.09
...,...,...,...
143,whole eggs,21.3,0.15
144,whole milk,214.6,1.62
145,yellow bell pepper,14.3,0.11
146,zucchini,5.3,0.01


In [28]:
daily_nutrition_df = planner.get_daily_nutrition()
daily_nutrition_df

,Calories,Protein,Δ Calories and Target Calories,Δ Protein and Target Protein
Day 1,1988.3 kcal,52.4 g,-11.7 kcal,2.4 g
Day 2,1987.6 kcal,47.8 g,-12.4 kcal,-2.2 g
Day 3,1928.5 kcal,50.0 g,-71.5 kcal,0.0 g
Day 4,2016.0 kcal,50.8 g,16.0 kcal,0.8 g
Day 5,2011.9 kcal,49.4 g,11.9 kcal,-0.6 g
Day 6,1985.2 kcal,48.1 g,-14.8 kcal,-1.9 g
Day 7,2009.6 kcal,49.2 g,9.6 kcal,-0.8 g
